# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1.
2.
3.
4.
5.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [ ]:
# База

In [4]:
import pandas as pd

df = pd.read_csv('orders.csv')
df

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00,C00557,Астана,Электроника,22150.0,NaN,оплачен
5461,103460,2026-10-31 22:00,C02162,Алматы,Электроника,24670.0,2.0,отменён


In [12]:
def to_num(s):
    s = str(s).strip()
    s = s.replace(" ", "")
    s = "".join(s.split())
    s = s.replace(",", ".")

    return float(s)

def clean_city(s):
    s = str(s).strip().lower()

    cities = {
        'астана': 'Астана',
        'astana': 'Астана',
        'алматы': 'Алматы',
        'almaty': 'Алматы',
        'шымкент': 'Шымкент',
        'shymkent': 'Шымкент',
        'караганда': 'Караганда',
        'актобе': 'Актобе',
        'атырау': 'Атырау'
    }

    return cities.get(s, s.capitalize())

In [13]:
df['amount'] = df['amount'].apply(to_num)
df['city'] = df['city'].apply(clean_city)
print(df['city'].value_counts())

city
Алматы       2188
Астана       1390
Шымкент       804
Караганда     538
Актобе        397
Атырау        146
Name: count, dtype: int64


In [20]:
category_count = df['category'].value_counts()

print(category_count)

category
Одежда         1529
Электроника    1185
Дом            1105
Красота         991
Книги           653
Name: count, dtype: int64


In [15]:
paid = df[df['status'] == 'оплачен']

result = paid.groupby('category')['amount'].agg(
    revenue='sum',
    average_check='mean'
)

print(result)

                revenue  average_check
category                              
Дом          10516320.0   10654.832827
Книги         2294100.0    3934.991424
Красота       5507620.0    6188.337079
Одежда       18321080.0   13285.772299
Электроника  65270240.0   61286.610329


In [21]:
df['month'] = df['order_date'].str[:7]

monthly_count = df['month'].value_counts().sort_index()

print(monthly_count)

month
2026-09    2782
2026-10    2681
Name: count, dtype: int64


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт

In [24]:
monthly_revenue = {}

for date, amount, status in zip(
    df["order_date"], df["amount"], df["status"]
):
    if status == "оплачен":
        month = date[:7]

        monthly_revenue[month] = monthly_revenue.get(month, 0) + amount
print("REVENUE BY MONTH")

for month in sorted(monthly_revenue):
    print(f"{month}: {monthly_revenue[month]:.2f}")
september = monthly_revenue.get("2026-09", 0)
october = monthly_revenue.get("2026-10", 0)

if september != 0:
    change = (october - september) / september * 100
    print(f"October change: {change:.1f}%")

REVENUE BY MONTH
2026-09: 57489850.00
2026-10: 44419510.00
October change: -22.7%


In [25]:
city_total = {}
city_cancelled = {}
for city, status in zip(df["city"], df["status"]):
    city_total[city] = city_total.get(city, 0) + 1
    if status == "отменён":
        city_cancelled[city] = city_cancelled.get(city, 0) + 1
print("CANCELLED ORDERS BY CITY")

for city in city_total:
    total = city_total[city]
    cancelled = city_cancelled.get(city, 0)
    percent = cancelled / total * 100
    print(f"{city}: {percent:.1f}%")

CANCELLED ORDERS BY CITY
Алматы: 7.9%
Астана: 8.8%
Караганда: 8.9%
Шымкент: 20.4%
Атырау: 7.5%
Актобе: 10.3%


In [26]:
customer_revenue = {}

for customer, amount, status in zip(
    df["customer_id"], df["amount"], df["status"]
):
    if status == "оплачен":
        customer_revenue[customer] = customer_revenue.get(customer, 0) + amount
top_customers = list(customer_revenue.items())
for i in range(len(top_customers)):
    for j in range(i + 1, len(top_customers)):

        if top_customers[j][1] > top_customers[i][1]:
            top_customers[i], top_customers[j] = top_customers[j], top_customers[i]
print("TOP 5")
for customer, revenue in top_customers[:5]:
    print(f"{customer}: {revenue:.2f}")

TOP 5
TEST: 29999970.00
C01627: 257180.00
C00485: 215920.00
C01338: 192080.00
C02122: 191670.00


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж